### Database creation for piano log project

In [13]:
import sqlite3

conn = sqlite3.connect("piano.db")
cursor = conn.cursor()

# --- COMPOSERS ---
cursor.execute("""
CREATE TABLE IF NOT EXISTS composers (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    normalized_name TEXT UNIQUE
)
""")

# --- PIECES ---
cursor.execute("""
CREATE TABLE IF NOT EXISTS pieces (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    canonical_name TEXT NOT NULL,
    normalized_name TEXT,
    composer_id INTEGER,
    opus TEXT,
    number TEXT,
    key TEXT,

    UNIQUE(canonical_name, composer_id),

    FOREIGN KEY (composer_id) REFERENCES composers(id)
)
""")

# --- ALIASES ---
cursor.execute("""
CREATE TABLE IF NOT EXISTS piece_aliases (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    alias TEXT NOT NULL,
    normalized_alias TEXT,
    piece_id INTEGER,

    FOREIGN KEY (piece_id) REFERENCES pieces(id)
)
""")

# --- SESSIONS ---
cursor.execute("""
CREATE TABLE IF NOT EXISTS sessions (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    date TEXT NOT NULL,
    duration_min INTEGER,
    raw_text TEXT
)
""")

# --- ACTIVITIES ---
cursor.execute("""
CREATE TABLE IF NOT EXISTS activities (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    session_id INTEGER NOT NULL,

    -- warmup | piece | improvisation | sight_reading
    type TEXT CHECK (
        type IN ('warmup', 'piece', 'improvisation', 'sight_reading')
    ),

    -- repertoire metadata (optional)
    piece_id INTEGER,
    composer_id INTEGER,
    piece_name TEXT,
    composer_name TEXT,
    key TEXT,
    section TEXT,
    bars TEXT,

    -- warmup/exercise metadata (optional)
    exercise_name TEXT,
    tempo INTEGER,
    repetitions INTEGER,

    -- general metadata
    focus TEXT,
    notes TEXT,

    FOREIGN KEY (session_id) REFERENCES sessions(id),
    FOREIGN KEY (piece_id) REFERENCES pieces(id),
    FOREIGN KEY (composer_id) REFERENCES composers(id)
)
""")

conn.commit()


### Helper functions

In [3]:
def normalize(text):
    return text.lower().strip() if text else None

## Composer Lookup

In [4]:
def get_or_create_composer(name):
    if not name:
        return None

    norm = normalize(name)

    cursor.execute(
        "SELECT id FROM composers WHERE normalized_name = ?",
        (norm,)
    )
    row = cursor.fetchone()
    if row:
        return row[0]

    cursor.execute(
        "INSERT INTO composers (name, normalized_name) VALUES (?, ?)",
        (name, norm)
    )
    return cursor.lastrowid

## Piece Lookup

In [5]:
def get_or_create_piece(piece_name, composer_id):
    if not piece_name:
        return None

    norm = normalize(piece_name)

    cursor.execute("""
        SELECT id FROM pieces
        WHERE normalized_name = ? AND composer_id IS ?
    """, (norm, composer_id))

    row = cursor.fetchone()
    if row:
        return row[0]

    cursor.execute("""
        INSERT INTO pieces (canonical_name, normalized_name, composer_id)
        VALUES (?, ?, ?)
    """, (piece_name, norm, composer_id))

    return cursor.lastrowid


## Session update

In [6]:
def log_session(session):
    # Insert session
    cursor.execute(
        "INSERT INTO sessions (date, duration_min, raw_text) VALUES (?, ?, ?)",
        (session["date"], session["duration_min"], session["raw_text"])
    )
    session_id = cursor.lastrowid

    # Insert activities
    for act in session["activities"]:
        composer_id = get_or_create_composer(act.get("composer_name"))
        piece_id = get_or_create_piece(act.get("piece_name"), composer_id)

        cursor.execute("""
            INSERT INTO activities (
                session_id, type,
                piece_id, composer_id,
                piece_name, composer_name,
                key, section, bars,
                exercise_name, tempo, repetitions,
                focus, notes
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            session_id,
            act.get("type"),

            piece_id,
            composer_id,

            act.get("piece_name"),
            act.get("composer_name"),

            act.get("key"),
            act.get("section"),
            act.get("bars"),

            act.get("exercise_name"),
            act.get("tempo"),
            act.get("repetitions"),

            act.get("focus"),
            act.get("notes"),
        ))

    conn.commit()

## ADD sessions

In [7]:
session1 = {
    "date": "2026-04-15",
    "duration_min": 50,
    "raw_text": "15 apr 26,50 min, warmup scales on D# minor, D major, weakfingers, worked on chopin fantasie impromptu op 66 c# minor bars [45-50, 72-80], improv on A minor",
    "activities": [
        {
            "type": "warmup",
            "exercise_name": "scales",
            "key": "D# minor, D major",
            "focus": "weak fingers",
            "notes": None,
            "piece_name": None,
            "composer_name": None,
            "bars": None,
            "section": None,
            "tempo": None,
            "repetitions": None
        },
        {
            "type": "piece",
            "piece_name": "Fantasie Impromptu Op. 66",
            "composer_name": "Chopin",
            "key": "C# minor",
            "bars": "45-50, 72-80",
            "focus": "left hand balance, memory, speed",
            "notes": None,
            "exercise_name": None,
            "tempo": None,
            "repetitions": None,
            "section": None
        },
        {
            "type": "improvisation",
            "key": "A minor",
            "piece_name": None,
            "composer_name": None,
            "bars": None,
            "section": None,
            "exercise_name": None,
            "tempo": None,
            "repetitions": None,
            "focus": None,
            "notes": None
        }
    ]
}

In [8]:
session2 = {
    "date": "2026-04-16",
    "duration_min": 30,
    "raw_text": "16 apr 26, 30 min, warmup arpeggios in G major, slow tempo, focus on evenness",
    "activities": [
        {
            "type": "warmup",
            "exercise_name": "arpeggios",
            "key": "G major",
            "tempo": 60,
            "repetitions": 5,
            "focus": "evenness",
            "notes": None,
            "piece_name": None,
            "composer_name": None,
            "bars": None,
            "section": None
        }
    ]
}

In [9]:
session3 = {
    "date": "2026-04-17",
    "duration_min": 40,
    "raw_text": "17 apr 26, 40 min, sight reading bach inventions, worked on beethoven sonata op 27 no 2 adagio sostenuto",
    "activities": [
        {
            "type": "sight_reading",
            "piece_name": "Two-Part Inventions",
            "composer_name": "Bach",
            "key": None,
            "bars": None,
            "section": None,
            "exercise_name": None,
            "tempo": None,
            "repetitions": None,
            "focus": None,
            "notes": None
        },
        {
            "type": "piece",
            "piece_name": "Sonata Op. 27 No. 2",
            "composer_name": "Beethoven",
            "section": "Adagio sostenuto",
            "key": "C# minor",
            "bars": None,
            "exercise_name": None,
            "tempo": None,
            "repetitions": None,
            "focus": "tone control",
            "notes": None
        }
    ]
}


## test inserting sessions

In [12]:
log_session(session1)
log_session(session2)
log_session(session3)

print("Inserted 3 test sessions successfully.")

Inserted 3 test sessions successfully.
